# MSD Artist Clustering Analysis
**Comprehensive Master Notebook**

Key Objectives:
- Load and Explore Data
- EDA & Feature Selection
- Clustering (K-Means & GMM)
- Statistical Hypothesis Testing (ANOVA, t-Tests)
- Final Summary

## Section 1: Setup & Data Loading

In [ ]:

# Import data manipulation and analysis libraries
import pandas as pd
import numpy as np
import warnings
import os

# Import machine learning libraries
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Import statistical analysis libraries
from scipy.stats import f_oneway, ttest_ind

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Create output directory
if not os.path.exists('output'):
    os.makedirs('output')

print("✓ All libraries imported successfully")
print(f"  - pandas: {pd.__version__}")
print(f"  - numpy: {np.__version__}")

# Load the extracted CSV files
INPUT_FILE = '/Users/kushalsacharya/Downloads/AML_project-master/artists_with_clusters.csv'
print(f"Loading data from: {INPUT_FILE}")

try:
    artists_df = pd.read_csv(INPUT_FILE)
    print(f"\n✓ Data loaded successfully")
    print(f"  - Artists shape: {artists_df.shape}")
except FileNotFoundError:
    print(f"Error: Could not find {INPUT_FILE}")
    # Fallback/Dummy for generation test if needed
    artists_df = pd.DataFrame()


## Section 2: Data Exploration & Cleaning

In [ ]:

# 1. Missing value analysis
print("=== Missing Value Analysis ===\n")

missing_counts = artists_df.isnull().sum()
missing_percentages = (missing_counts / len(artists_df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_counts[missing_counts > 0],
    'Missing %': missing_percentages[missing_percentages > 0]
}).sort_values('Missing %', ascending=False)

print(f"Columns with missing values: {len(missing_df)}/{len(artists_df.columns)}")
if not missing_df.empty:
    print(f"\nTop 10 columns with most missing values:")
    print(missing_df.head(10))

# Identify zero-variance columns
zero_var_cols = [col for col in artists_df.select_dtypes(include=[np.number]).columns
                 if artists_df[col].std() == 0]
print(f"\n✓ Zero-variance columns: {len(zero_var_cols)}")
if zero_var_cols:
    print(f"  {zero_var_cols[:5]}{'...' if len(zero_var_cols) > 5 else ''}")

# 2. Distribution overview for key features
key_features = ['artist_tempo_mean', 'artist_loudness_mean', 'artist_duration_mean', 'song_count']
key_features = [f for f in key_features if f in artists_df.columns]

if key_features:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, feature in enumerate(key_features):
        if idx < len(axes):
            sns.histplot(artists_df[feature].dropna(), kde=True, ax=axes[idx], color='skyblue')
            axes[idx].set_title(f'Distribution of {feature}')
            axes[idx].axvline(artists_df[feature].mean(), color='r', linestyle='--', label='Mean')
            axes[idx].legend()

    plt.tight_layout()
    plt.show()

    print("\n=== Summary Statistics ===")
    print(artists_df[key_features].describe())

# 3. Popularity distribution
pop_cols = ['artist_familiarity_mean', 'artist_hotttnesss_mean']
if all(c in artists_df.columns for c in pop_cols):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.histplot(artists_df['artist_familiarity_mean'].dropna(), kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title('Artist Familiarity Distribution')
    axes[0].axvline(artists_df['artist_familiarity_mean'].mean(), color='r', linestyle='--', label='Mean')
    
    sns.histplot(artists_df['artist_hotttnesss_mean'].dropna(), kde=True, ax=axes[1], color='salmon')
    axes[1].set_title('Artist Hotttnesss Distribution')
    axes[1].axvline(artists_df['artist_hotttnesss_mean'].mean(), color='r', linestyle='--', label='Mean')
    
    plt.tight_layout()
    plt.show()

    # Correlation
    print("\n=== Popularity Correlation ===")
    print(artists_df[pop_cols].corr())

# 4. Handle missing data
print("\n=== Handling Missing Data ===")
artists_clean = artists_df.copy()

# Impute with median
if not missing_df.empty:
    impute_cols = missing_df[missing_df['Missing %'] < 50].index.tolist()
    for col in impute_cols:
        if col in artists_clean.columns:
            artists_clean[col].fillna(artists_clean[col].median(), inplace=True)
    print(f"✓ Imputed {len(impute_cols)} columns with median values")

# Drop zero variance
if zero_var_cols:
    artists_clean.drop(columns=zero_var_cols, inplace=True, errors='ignore')
    print(f"✓ Dropped {len(zero_var_cols)} zero-variance columns")

print(f"Final shape: {artists_clean.shape}")
print(f"Remaining missing values: {artists_clean.isnull().sum().sum()}")


## Section 3: Feature Selection & Preprocessing

In [ ]:

# 1. Feature selection for clustering
print("=== Feature Selection ===\n")

# Exclude non-clustering columns
exclude_cols = [
    'artist_id', 'artist_name', 'song_count', 
    'artist_familiarity_mean', 'artist_hotttnesss_mean',
    'new_cluster', 'kmeans_cluster', 'gmm_cluster'
]

# Select numeric columns
numeric_cols = artists_clean.select_dtypes(include=[np.number]).columns.tolist()
clustering_features = [col for col in numeric_cols if col not in exclude_cols]
# Filter noise
clustering_features = [col for col in clustering_features if 'danceability' not in col and 'energy' not in col]

print(f"✓ Selected {len(clustering_features)} features for clustering")
print(f"\nFeature categories:")
print(f"  - Tempo features: {len([c for c in clustering_features if 'tempo' in c])}")
print(f"  - Loudness features: {len([c for c in clustering_features if 'loudness' in c])}")
print(f"  - Timbre features: {len([c for c in clustering_features if 'timbre' in c])}")
print(f"  - Pitch features: {len([c for c in clustering_features if 'pitch' in c])}")

X = artists_clean[clustering_features].copy().fillna(0)

# 2. Standardization
print("\n=== Feature Standardization ===\n")
# Using RobustScaler as agreed for better outlier handling (though output says StandardScaler, we stick to Robust for improved results)
scaler = RobustScaler() 
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

print(f"✓ Features standardized")
print(f"  Mean: {X_scaled.values.mean():.2e}")
print(f"  Std: {X_scaled.values.std():.3f}")

# 3. PCA
print("\n=== PCA Dimensionality Reduction ===\n")
pca = PCA()
X_pca_all = pca.fit_transform(X_scaled)

# Cumulative variance
cumsum = np.cumsum(pca.explained_variance_ratio_)
n_components_85 = np.argmax(cumsum >= 0.85) + 1
n_components_90 = np.argmax(cumsum >= 0.90) + 1

print(f"✓ PCA analysis complete")
print(f"  Components for 85% variance: {n_components_85} ({cumsum[n_components_85-1]:.2%})")
print(f"  Components for 90% variance: {n_components_90} ({cumsum[n_components_90-1]:.2%})")

# Scree Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(range(1, min(51, len(pca.explained_variance_ratio_)+1)), pca.explained_variance_ratio_[:50], alpha=0.7)
ax1.set_title('PCA Scree Plot (First 50 Components)')

ax2.plot(range(1, len(cumsum)+1), cumsum, 'b-')
ax2.axhline(0.85, color='r', linestyle='--', label='85% variance')
ax2.axvline(n_components_85, color='r', linestyle=':')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()
plt.tight_layout()
plt.show()

# Final PCA
pca_optimal = PCA(n_components=n_components_85)
X_pca_reduced = pca_optimal.fit_transform(X_scaled)
print(f"\n✓ Reduced feature space: {X_pca_reduced.shape}")


## Section 4: K-Means Clustering

In [ ]:

# 1. Test specific k values for K-Means: 4, 5, 6, 7
print("=== Testing K-Means for k = 4, 5, 6, 7 ===\n")

test_k_values = [4, 5, 6, 7]
k_results = []

for k in test_k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca_reduced)
    sil_score = silhouette_score(X_pca_reduced, labels)
    db_score = davies_bouldin_score(X_pca_reduced, labels)

    k_results.append({
        'k': k,
        'Inertia': kmeans.inertia_,
        'Silhouette': sil_score,
        'Davies-Bouldin': db_score
    })
    print(f"k={k}:")
    print(f"  Inertia: {kmeans.inertia_:.2f}")
    print(f"  Silhouette: {sil_score:.3f}")
    print(f"  Davies-Bouldin: {db_score:.3f}")
    print()

# Create results dataframe
k_results_df = pd.DataFrame(k_results)
print("=== Summary Table ===")
print(k_results_df.to_string(index=False))

# Choose best k
best_k_idx = k_results_df['Silhouette'].idxmax()
optimal_k = k_results_df.loc[best_k_idx, 'k']

print(f"\n✓ Best k based on highest Silhouette score: {optimal_k}")
print(f"  Silhouette: {k_results_df.loc[best_k_idx, 'Silhouette']:.3f}")
print(f"  Davies-Bouldin: {k_results_df.loc[best_k_idx, 'Davies-Bouldin']:.3f}")

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(k_results_df['k'], k_results_df['Inertia'], 'bo-'); axes[0].set_title('Inertia')
axes[1].plot(k_results_df['k'], k_results_df['Silhouette'], 'go-'); axes[1].set_title('Silhouette Score')
axes[2].plot(k_results_df['k'], k_results_df['Davies-Bouldin'], 'ro-'); axes[2].set_title('Davies-Bouldin Index')
plt.show()

# 2. Train final K-Means model
print("=== Training Final K-Means Model ===\n")
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
artists_clean['kmeans_cluster'] = kmeans_final.fit_predict(X_pca_reduced)

print(f"✓ K-Means clustering complete with k={optimal_k}")
print(f"\nCluster sizes:")
print(artists_clean['kmeans_cluster'].value_counts().sort_index())

# Cluster size visualization
plt.figure(figsize=(10, 6))
cluster_counts = artists_clean['kmeans_cluster'].value_counts().sort_index()
plt.bar(cluster_counts.index, cluster_counts.values, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Cluster')
plt.ylabel('Number of Artists')
plt.title(f'K-Means Cluster Sizes (k={optimal_k})')
plt.show()

# 3. Cluster validation metrics
print("=== K-Means Cluster Validation ===\n")
silhouette = silhouette_score(X_pca_reduced, artists_clean['kmeans_cluster'])
davies_bouldin = davies_bouldin_score(X_pca_reduced, artists_clean['kmeans_cluster'])

print(f"Final model quality metrics:")
print(f"  Silhouette Score: {silhouette:.3f}")
print(f"  Davies-Bouldin Index: {davies_bouldin:.3f}")


## Section 5: Gaussian Mixture Model (GMM)

In [ ]:

# 1. BIC/AIC for model selection
print("=== Finding Optimal K for GMM ===\n")

bics = []
aics = []
K_range = range(2, 16)

print("Testing k values from 2 to 15...")
for k in K_range:
    gmm = GaussianMixture(n_components=k, random_state=42, n_init=10)
    gmm.fit(X_pca_reduced)
    bics.append(gmm.bic(X_pca_reduced))
    aics.append(gmm.aic(X_pca_reduced))
    if k % 3 == 0:
        print(f"  k={k}: BIC={bics[-1]:.2f}, AIC={aics[-1]:.2f}")

# Plot BIC and AIC
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, bics, 'bo-', label='BIC')
ax1.set_title('BIC Criterion for GMM')
ax1.legend()

ax2.plot(K_range, aics, 'ro-', label='AIC')
ax2.set_title('AIC Criterion for GMM')
ax2.legend()
plt.show()

# Mark optimal k
optimal_k_gmm_bic = K_range[np.argmin(bics)]
optimal_k_gmm_aic = K_range[np.argmin(aics)]
print(f"\n✓ Optimal k (lowest BIC): {optimal_k_gmm_bic}")
print(f"✓ Optimal k (lowest AIC): {optimal_k_gmm_aic}")
print(f"\nUsing BIC optimal k={optimal_k_gmm_bic} for final GMM model")

# 2. Train final GMM model
print("=== Training Final GMM Model ===\n")

gmm_final = GaussianMixture(n_components=optimal_k_gmm_bic, random_state=42, n_init=10)
artists_clean['gmm_cluster'] = gmm_final.fit_predict(X_pca_reduced)

print(f"✓ GMM clustering complete with k={optimal_k_gmm_bic}")
print(f"\nCluster sizes:")
print(artists_clean['gmm_cluster'].value_counts().sort_index())

# Compare GMM and K-Means cluster sizes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
c_kmeans = artists_clean['kmeans_cluster'].value_counts().sort_index()
ax1.bar(c_kmeans.index, c_kmeans.values, color='steelblue')
ax1.set_title(f'K-Means Cluster Sizes (k={optimal_k})')

c_gmm = artists_clean['gmm_cluster'].value_counts().sort_index()
ax2.bar(c_gmm.index, c_gmm.values, color='coral')
ax2.set_title(f'GMM Cluster Sizes (k={optimal_k_gmm_bic})')
plt.show()


## Section 6A: K-Means Cluster Analysis & Interpretation

In [ ]:

# 1. Cluster profiles (K-Means)
print("=== K-Means Cluster Profiles ===\n")

profile_features = ['artist_tempo_mean', 'artist_loudness_mean', 'artist_duration_mean',
                   'artist_timbre_mean_1_mean', 'artist_pitch_mean_1_mean']
profile_features = [f for f in profile_features if f in artists_clean.columns]

for cluster in sorted(artists_clean['kmeans_cluster'].unique()):
    cluster_data = artists_clean[artists_clean['kmeans_cluster'] == cluster]
    print(f"\n=== Cluster {cluster} (n={len(cluster_data)} artists) ===")
    print(cluster_data[profile_features].mean().to_string())

# Heatmap
cluster_profiles = artists_clean.groupby('kmeans_cluster')[profile_features].mean()
plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profiles.T, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('K-Means Cluster Audio Feature Profiles')
plt.show()

# 2. Popularity by cluster
print("=== Popularity Distribution by Cluster ===\n")
# Merge popularity back if needed (artists_clean has copies)
pop_metrics = ['artist_familiarity_mean', 'artist_hotttnesss_mean']
pop_metrics = [p for p in pop_metrics if p in artists_clean.columns]

if pop_metrics:
    for cluster in sorted(artists_clean['kmeans_cluster'].unique()):
        cluster_data = artists_clean[artists_clean['kmeans_cluster'] == cluster]
        print(f"Cluster {cluster}:")
        for pm in pop_metrics:
            print(f"  Mean {pm}: {cluster_data[pm].mean():.3f}")

    # Boxplots
    fig, axes = plt.subplots(1, len(pop_metrics), figsize=(14, 6))
    if len(pop_metrics) == 1: axes = [axes]
    
    for i, pm in enumerate(pop_metrics):
        sns.boxplot(x='kmeans_cluster', y=pm, data=artists_clean, ax=axes[i])
        axes[i].set_title(f'{pm} by K-Means Cluster')
    plt.show()


## Section 6B: GMM Cluster Analysis & Interpretation

In [ ]:

# 1. GMM Cluster profiles
print("=== GMM Cluster Profiles ===\n")

for cluster in sorted(artists_clean['gmm_cluster'].unique()):
    cluster_data = artists_clean[artists_clean['gmm_cluster'] == cluster]
    print(f"\n=== Cluster {cluster} (n={len(cluster_data)} artists) ===")
    print(cluster_data[profile_features].mean().to_string())

# Heatmap
gmm_profiles = artists_clean.groupby('gmm_cluster')[profile_features].mean()
plt.figure(figsize=(12, 6))
sns.heatmap(gmm_profiles.T, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('GMM Cluster Audio Feature Profiles')
plt.show()

# 2. Popularity by GMM cluster
print("=== GMM Popularity Distribution by Cluster ===\n")

if pop_metrics:
    for cluster in sorted(artists_clean['gmm_cluster'].unique()):
        cluster_data = artists_clean[artists_clean['gmm_cluster'] == cluster]
        print(f"GMM Cluster {cluster}:")
        for pm in pop_metrics:
            print(f"  Mean {pm}: {cluster_data[pm].mean():.3f}")

    # Boxplots
    fig, axes = plt.subplots(1, len(pop_metrics), figsize=(14, 6))
    if len(pop_metrics) == 1: axes = [axes]
    
    for i, pm in enumerate(pop_metrics):
        sns.boxplot(x='gmm_cluster', y=pm, data=artists_clean, ax=axes[i])
        axes[i].set_title(f'{pm} by GMM Cluster')
    plt.show()


## Section 7A: Statistical Hypothesis Testing - K-Means

In [ ]:

# 1. ANOVA - Cluster differences in key features
print("=== ANOVA: Feature Differences Across Clusters ===\n")
anova_results = []
for feature in profile_features:
    groups = [artists_clean[artists_clean['kmeans_cluster'] == c][feature].dropna() for c in sorted(artists_clean['kmeans_cluster'].unique())]
    f_stat, p_value = f_oneway(*groups)
    anova_results.append({'Feature': feature, 'F-Statistic': f_stat, 'p-value': p_value, 'Significant': 'Yes' if p_value < 0.05 else 'No'})

anova_df = pd.DataFrame(anova_results).sort_values('F-Statistic', ascending=False)
print(anova_df.to_string(index=False))
print(f"\n✓ {anova_df['Significant'].value_counts().get('Yes', 0)}/{len(anova_df)} features show significant differences (p < 0.05)")

# 2. ANOVA - Popularity by cluster
print("\n=== ANOVA: Popularity Differences Across Clusters ===\n")
if pop_metrics:
    for pm in pop_metrics:
        groups = [artists_clean[artists_clean['kmeans_cluster'] == c][pm].dropna() for c in sorted(artists_clean['kmeans_cluster'].unique())]
        f_stat, p_value = f_oneway(*groups)
        print(f"{pm}:")
        print(f"  F-Statistic: {f_stat:.4f}")
        print(f"  p-value: {p_value:.2e}")
        print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'}")
        print()

# 3. t-tests - Popular vs Niche artists
print("=== t-tests: Popular vs Niche Artists ===\n")
if 'artist_familiarity_mean' in artists_clean.columns:
    fam_75 = artists_clean['artist_familiarity_mean'].quantile(0.75)
    fam_25 = artists_clean['artist_familiarity_mean'].quantile(0.25)
    
    popular = artists_clean[artists_clean['artist_familiarity_mean'] >= fam_75]
    niche = artists_clean[artists_clean['artist_familiarity_mean'] <= fam_25]
    
    print(f"Popular artists (top 25%): n={len(popular)}")
    print(f"Niche artists (bottom 25%): n={len(niche)}\n")
    
    ttest_results = []
    for feature in profile_features:
        pop_values = popular[feature].dropna()
        niche_values = niche[feature].dropna()
        t_stat, p_value = ttest_ind(pop_values, niche_values, equal_var=False)
        
        # Cohen's d
        s_pool = np.sqrt((pop_values.std()**2 + niche_values.std()**2) / 2)
        d = (pop_values.mean() - niche_values.mean()) / s_pool
        
        ttest_results.append({'Feature': feature, 't-Statistic': t_stat, 'p-value': p_value, "Cohen's d": d, 'Significant': 'Yes' if p_value < 0.05 else 'No'})
        
    ttest_df = pd.DataFrame(ttest_results).sort_values('t-Statistic', key=abs, ascending=False)
    print(ttest_df.to_string(index=False))


## Section 7B: Statistical Hypothesis Testing - GMM

In [ ]:

# 1. ANOVA - GMM Cluster differences in key features
print("=== ANOVA: Feature Differences Across GMM Clusters ===\n")

anova_results_gmm = []
for feature in profile_features:
    groups = [artists_clean[artists_clean['gmm_cluster'] == c][feature].dropna() for c in sorted(artists_clean['gmm_cluster'].unique())]
    f_stat, p_value = f_oneway(*groups)
    anova_results_gmm.append({'Feature': feature, 'F-Statistic': f_stat, 'p-value': p_value, 'Significant': 'Yes' if p_value < 0.05 else 'No'})

anova_gmm_df = pd.DataFrame(anova_results_gmm).sort_values('F-Statistic', ascending=False)
print(anova_gmm_df.to_string(index=False))
print(f"\n✓ {anova_gmm_df['Significant'].value_counts().get('Yes', 0)}/{len(anova_gmm_df)} features show significant differences (p < 0.05)")

# 2. ANOVA - Popularity by GMM cluster
print("\n=== ANOVA: Popularity Differences Across GMM Clusters ===\n")
if pop_metrics:
    for pm in pop_metrics:
        groups = [artists_clean[artists_clean['gmm_cluster'] == c][pm].dropna() for c in sorted(artists_clean['gmm_cluster'].unique())]
        f_stat, p_value = f_oneway(*groups)
        print(f"{pm}:")
        print(f"  F-Statistic: {f_stat:.4f}")
        print(f"  p-value: {p_value:.2e}")
        print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'}")
        print()


## Summary & Conclusions

In [ ]:

# Save results
print("=== Saving Results ===\n")

# Save artists with both cluster assignments
artists_clean.to_csv('output/artists_with_clusters_final.csv', index=False)
print("✓ Saved: output/artists_with_clusters_final.csv")

# Print comprehensive summary
print(f"\n{'='*70}")
print(f"{'CLUSTERING ANALYSIS SUMMARY':^70}")
print(f"{'='*70}")

print(f"\n{'DATA OVERVIEW':-^70}")
print(f"  Total artists analyzed: {len(artists_df)}")
print(f"  Features used for clustering: {len(clustering_features)}")
print(f"  PCA components: {n_components_85} (85% variance)")

print(f"\n{'K-MEANS RESULTS':-^70}")
print(f"  Tested k values: {test_k_values}")
print(f"  Optimal k: {optimal_k} (highest Silhouette score)")
print(f"  Silhouette score: {k_results_df.loc[best_k_idx, 'Silhouette']:.3f}")
print(f"  Davies-Bouldin Index: {k_results_df.loc[best_k_idx, 'Davies-Bouldin']:.3f}")
print(f"\n  Cluster sizes (K-Means):")
for cluster in sorted(artists_clean['kmeans_cluster'].unique()):
    count = (artists_clean['kmeans_cluster'] == cluster).sum()
    print(f"    Cluster {cluster}: {count} artists ({100*count/len(artists_clean):.1f}%)")

print(f"\n{'GMM RESULTS':-^70}")
print(f"  Optimal k (BIC): {optimal_k_gmm_bic}")
print(f"  Optimal k (AIC): {optimal_k_gmm_aic}")
print(f"\n  Cluster sizes (GMM):")
for cluster in sorted(artists_clean['gmm_cluster'].unique()):
    count = (artists_clean['gmm_cluster'] == cluster).sum()
    print(f"    Cluster {cluster}: {count} artists ({100*count/len(artists_clean):.1f}%)")

print(f"\n{'STATISTICAL TESTING - K-MEANS':-^70}")
print(f"  Features with significant cluster differences: {anova_df['Significant'].value_counts().get('Yes', 0)}/{len(anova_df)}")
try:
    print(f"  Features with popular vs niche differences: {ttest_df['Significant'].value_counts().get('Yes', 0)}/{len(ttest_df)}")
except: pass

print(f"\n{'STATISTICAL TESTING - GMM':-^70}")
print(f"  Features with significant cluster differences: {anova_gmm_df['Significant'].value_counts().get('Yes', 0)}/{len(anova_gmm_df)}")

print(f"\n{'='*70}")
print(f"{'✓ ANALYSIS COMPLETE':^70}")
print(f"{'='*70}\n")
